## Part 4

**Same article scraper, re-run with resume settings.** Identical fetch/parse/log logic as Part 3, but it reads the URL lists from `outputs_urls/` and picks up where an interrupted run stopped: `START_FROM` skips every earlier month outright, and inside that month `RESUME_AFTER_URL` skips all URLs up to and including that one before continuing.

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import os
import glob
import json
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════
#  MALAY MAIL SCRAPER  (2015–2025, all months)
#  Mirrors Bernama v3: all log fields saved as columns in the same CSV
# ═══════════════════════════════════════════════════════════════════════

# ───────────────── INPUT / OUTPUT ─────────────────

INPUT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\outputs_urls"

# ── RESUME SETTINGS ──
# Start from this CSV file (skips all earlier months entirely)
START_FROM = "malaymail_2021_04.csv"

# Within START_FROM month, resume AFTER this URL (set to "" to scrape all URLs in that month)
# The scraper will skip all URLs up to AND INCLUDING this one, then continue from the next.
RESUME_AFTER_URL = "https://www.malaymail.com/news/malaysia/2019/07/01/former-malaysia-pm-najibs-rm42m-src-trial-days-18-to-29-catch-up/1766939"

OUTPUT_DIR     = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\scraped"
CHECKPOINT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\checkpoints"

# ───────────────── RETRY / CHECKPOINT CONFIG ─────────────────

MAX_RETRIES      = 3
RETRY_DELAY      = 5
CHECKPOINT_EVERY = 50

# ───────────────── TEST MODE ─────────────────
TEST_MODE = False

# ───────────────── REQUEST HEADERS ─────────────────

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

# ───────────────── STATUS CODE MESSAGES ─────────────────

STATUS_MESSAGES = {
    200: "OK",
    301: "Moved Permanently",
    302: "Found (Redirect)",
    403: "Forbidden",
    404: "Not Found",
    429: "Too Many Requests",
    500: "Internal Server Error",
    503: "Service Unavailable",
}

def get_status_msg(code):
    return STATUS_MESSAGES.get(code, "Unknown Status")

# ───────────────── MALAY MAIL PARSER ─────────────────

def parse_article(html, url):
    soup = BeautifulSoup(html, "lxml")

    # ── TITLE ──
    title = ""
    h1 = soup.find("h1", class_="article-title")
    if h1:
        title = h1.get_text(" ", strip=True)
    elif soup.find("h1"):
        title = soup.find("h1").get_text(" ", strip=True)
    elif soup.title:
        title = soup.title.get_text(" ", strip=True).replace("| Malay Mail", "").strip()

    # ── PUBLISHED DATE + TIME ──
    published_date = ""
    published_time = ""
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date:
        raw = meta_date.get("content", "").strip()
        parts = raw.split(" ")
        if len(parts) >= 1:
            published_date = parts[0]
        if len(parts) >= 2:
            published_time = " ".join(parts[1:])

    # ── SECTION ──
    section = ""
    sec_div = soup.find("div", class_="article-section")
    if sec_div:
        sec_a = sec_div.find("a")
        if sec_a:
            section = sec_a.get_text(" ", strip=True)

    # ── AUTHOR ──
    author = ""
    meta_author = soup.find("meta", {"name": "author"})
    if meta_author:
        author = meta_author.get("content", "").strip()
    if not author:
        byline = soup.find("div", class_="article-byline")
        if byline:
            author = byline.get_text(" ", strip=True)

    # ── DESCRIPTION ──
    description = ""
    meta_desc = soup.find("meta", {"name": "description"})
    if meta_desc:
        description = meta_desc.get("content", "").strip()

    # ── KEYWORDS ──
    keywords = ""
    meta_kw = soup.find("meta", {"name": "keywords"})
    if meta_kw:
        keywords = meta_kw.get("content", "").strip()

    # ── IMAGE ──
    article_img = ""
    meta_img = soup.find("meta", {"property": "og:image"})
    if meta_img:
        article_img = meta_img.get("content", "").strip()

    # ── ARTICLE BODY TEXT ──
    paragraphs = []
    article_body = soup.find("div", class_="article-body")
    if article_body:
        for p in article_body.find_all("p"):
            text = p.get_text(" ", strip=True)
            if not text:
                continue
            bad_patterns = [
                "Follow us on",
                "Subscribe to",
                "You May Also Like",
                "Related Articles",
            ]
            if any(x in text for x in bad_patterns):
                continue
            paragraphs.append(text)
    else:
        for p in soup.find_all("p"):
            text = p.get_text(" ", strip=True)
            if text:
                paragraphs.append(text)

    paragraphs   = list(dict.fromkeys(paragraphs))
    article_text = "\n\n".join(paragraphs)

    return {
        "keyword":        "Malaysia",
        "published_date": published_date,
        "published_time": published_time,
        "section":        section,
        "author":         author,
        "title":          title,
        "description":    description,
        "keywords":       keywords,
        "article_text":   article_text,
        "article_img":    article_img,
        "url":            url,
    }

# ───────────────── CHECKPOINT SAVE ─────────────────

def save_checkpoint(rows, checkpoint_path):
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    pd.DataFrame(rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
    print(f"  [CHECKPOINT] {len(rows)} rows → {checkpoint_path}")

# ───────────────── COLLECT INPUT CSVs ─────────────────

csv_pattern   = os.path.join(INPUT_DIR, "malaymail_*.csv")
all_csv_files = sorted(glob.glob(csv_pattern))

# Skip all months before START_FROM
all_csv_files = [f for f in all_csv_files if os.path.basename(f) >= START_FROM]

if not all_csv_files:
    print(f"ERROR: No CSV files found matching: {csv_pattern}")
    exit(1)

print(f"\nFound {len(all_csv_files)} input CSV files (starting from {START_FROM})")
os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ───────────────── SCRIPT START TIME ─────────────────

script_start     = time.time()
script_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"SCRIPT STARTED: {script_start_str}")
if RESUME_AFTER_URL:
    print(f"RESUMING AFTER:  {RESUME_AFTER_URL}")

grand_total_rows = 0

# ═══════════════════════════════════════════════════════════════════════
#  OUTER LOOP — one CSV file at a time
# ═══════════════════════════════════════════════════════════════════════

for csv_idx, csv_path in enumerate(all_csv_files, start=1):

    csv_name = os.path.basename(csv_path)
    csv_stem = os.path.splitext(csv_name)[0]

    print("\n" + "█" * 70)
    print(f"[CSV {csv_idx}/{len(all_csv_files)}] {csv_name}")

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"  ERROR reading CSV: {e} — skipping")
        continue

    if "url" not in df.columns:
        print(f"  WARNING: no 'url' column in {csv_name} — skipping")
        continue

    urls = df["url"].dropna().unique().tolist()

    if TEST_MODE:
        if csv_idx == 1:
            urls = urls[:1]
            print(f"  TEST MODE: using 1 URL")
        else:
            print(f"  TEST MODE: skipping remaining CSVs")
            break

    # ── URL-LEVEL RESUME ──
    # For the FIRST csv file only (START_FROM month), skip all URLs
    # up to and including RESUME_AFTER_URL.
    # For all subsequent months, scrape every URL normally.
    if csv_idx == 1 and RESUME_AFTER_URL:
        if RESUME_AFTER_URL in urls:
            skip_until = urls.index(RESUME_AFTER_URL)
            skipped    = skip_until + 1   # +1 to also skip the resume URL itself
            urls       = urls[skipped:]
            print(f"  ♻️  Resuming: skipped {skipped} already-done URLs")
            print(f"  ♻️  Starting from URL index {skipped+1}: {urls[0] if urls else 'END'}")
        else:
            # URL not found — it may have been the last in the month,
            # so skip the entire first month and start fresh from next
            print(f"  ⚠️  RESUME_AFTER_URL not found in {csv_name}")
            print(f"  ⚠️  Skipping entire {csv_name} and starting from next month")
            continue

    print(f"  URLs to scrape: {len(urls)}")
    if not urls:
        print(f"  No URLs left in {csv_name} — moving to next month")
        continue

    # ── output paths ──
    # For the resume month, APPEND to existing output CSV if it exists
    output_csv        = os.path.join(OUTPUT_DIR,     f"{csv_stem}_scraped.csv")
    checkpoint_prefix = os.path.join(CHECKPOINT_DIR, f"{csv_stem}_ckpt")

    # Determine checkpoint index to avoid overwriting existing checkpoints
    existing_ckpts = glob.glob(f"{checkpoint_prefix}_*.csv")
    checkpoint_idx = len(existing_ckpts) + 1

    all_rows       = []
    checkpoint_buf = []
    total_urls     = len(urls)

    # ═══════════════════════════════════════════════════════════════════
    #  INNER LOOP — one URL at a time
    # ═══════════════════════════════════════════════════════════════════

    for url_idx, url in enumerate(urls, start=1):

        print("\n" + "=" * 70)
        print(f"  [{url_idx}/{total_urls}] FETCHING")
        print(f"  URL: {url}")

        response          = None
        attempt           = 0
        request_start_str = ""
        request_end_str   = ""
        request_duration  = 0.0
        status_code       = ""
        status_msg        = ""
        redirected        = "No"
        redirect_hops     = ""
        final_url         = url
        req_headers_str   = ""
        resp_headers_str  = ""
        wait_time         = 0.0
        total_attempts    = 0
        error_msg         = ""
        scrape_status     = "SKIPPED"

        while attempt < MAX_RETRIES:

            attempt += 1
            print(f"    ATTEMPT: {attempt}/{MAX_RETRIES}")

            request_start     = time.time()
            request_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"    REQUEST START: {request_start_str}")

            try:
                response = requests.get(
                    url,
                    headers=HEADERS,
                    timeout=30,
                    allow_redirects=True
                )

                request_end      = time.time()
                request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                request_duration = round(request_end - request_start, 2)

                print(f"    REQUEST END:   {request_end_str}")
                print(f"    REQUEST TIME:  {request_duration}s")

                status_code = response.status_code
                status_msg  = get_status_msg(status_code)
                print(f"    STATUS: {status_code} — {status_msg}")

                if response.history:
                    redirected    = f"Yes ({len(response.history)} hop(s))"
                    redirect_hops = " | ".join(
                        f"{h.status_code} → {h.url}" for h in response.history
                    )
                    final_url = response.url
                    print(f"    REDIRECTED: {redirected}")
                    for i, hop in enumerate(response.history, 1):
                        print(f"      hop {i}: {hop.status_code} → {hop.url}")
                    print(f"    FINAL URL: {final_url}")
                else:
                    print(f"    REDIRECTED: No")

                req_headers_str  = json.dumps(dict(response.request.headers), ensure_ascii=False)
                resp_headers_str = json.dumps(dict(response.headers),         ensure_ascii=False)

                print("    REQUEST HEADERS:")
                for k, v in response.request.headers.items():
                    print(f"      {k}: {v}")
                print("    RESPONSE HEADERS:")
                for k, v in response.headers.items():
                    print(f"      {k}: {v}")

                if status_code == 200:
                    break

                if status_code in (429, 500, 503):
                    print(f"    Retryable {status_code}. Waiting {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"    Non-retryable {status_code}. Skipping.")
                    break

            except Exception as e:
                request_end      = time.time()
                request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                request_duration = round(request_end - request_start, 2)
                error_msg        = str(e)
                scrape_status    = "ERROR"
                print(f"    ERROR on attempt {attempt}: {error_msg}")
                print(f"    REQUEST TIME (before error): {request_duration}s")
                if attempt < MAX_RETRIES:
                    print(f"    Waiting {RETRY_DELAY}s before retry...")
                    time.sleep(RETRY_DELAY)

        total_attempts = attempt

        if response is not None and response.status_code == 200:
            html    = response.text
            soup    = BeautifulSoup(html, "lxml")
            print(f"    TITLE CHECK: {soup.title}")
            print(f"    P TAGS: {len(soup.find_all('p'))}")
            article       = parse_article(html, url)
            print(f"    TITLE:        {article['title'][:100]}")
            print(f"    SECTION:      {article['section']}")
            print(f"    AUTHOR:       {article['author']}")
            print(f"    DATE:         {article['published_date']}")
            print(f"    TIME:         {article['published_time']}")
            print(f"    IMAGE FOUND:  {'Yes' if article['article_img'] else 'No'}")
            print(f"    TEXT CHARS:   {len(article['article_text'])}")
            scrape_status = "SUCCESS"
        else:
            article = {
                "keyword":        "Malaysia",
                "published_date": "",
                "published_time": "",
                "section":        "",
                "author":         "",
                "title":          "",
                "description":    "",
                "keywords":       "",
                "article_text":   "",
                "article_img":    "",
                "url":            url,
            }
            print(f"    SKIPPED (no valid response after {total_attempts} attempt(s))")

        wait_time = round(random.uniform(5, 10), 1)
        print(f"\n    WAITING {wait_time}s before next request...")
        time.sleep(wait_time)

        row = {
            **article,
            "log_request_start":      request_start_str,
            "log_request_end":        request_end_str,
            "log_request_duration_s": request_duration,
            "log_total_attempts":     total_attempts,
            "log_max_retries":        MAX_RETRIES,
            "log_wait_time_s":        wait_time,
            "log_status_code":        status_code,
            "log_status_msg":         status_msg,
            "log_redirected":         redirected,
            "log_redirect_hops":      redirect_hops,
            "log_final_url":          final_url,
            "log_request_headers":    req_headers_str,
            "log_response_headers":   resp_headers_str,
            "log_error_msg":          error_msg,
            "log_scrape_status":      scrape_status,
        }

        all_rows.append(row)
        checkpoint_buf.append(row)

        # ── checkpoint every N rows ──
        if len(checkpoint_buf) >= CHECKPOINT_EVERY:
            ckpt_path = f"{checkpoint_prefix}_{checkpoint_idx:04d}.csv"
            save_checkpoint(checkpoint_buf, ckpt_path)
            checkpoint_idx += 1
            checkpoint_buf  = []

    # ── save leftover rows from this month ──
    if checkpoint_buf:
        ckpt_path = f"{checkpoint_prefix}_{checkpoint_idx:04d}.csv"
        save_checkpoint(checkpoint_buf, ckpt_path)

    # ── save / append final output CSV for this month ──
    if all_rows:
        month_df      = pd.DataFrame(all_rows)
        file_exists   = os.path.exists(output_csv)

        if file_exists and csv_idx == 1:
            # Resume month → APPEND to existing file (don't overwrite done rows)
            month_df.to_csv(output_csv, mode="a", header=False,
                            index=False, encoding="utf-8-sig")
            print(f"\n  [APPENDED] {csv_name} → {output_csv}")
        else:
            # Fresh month → new file
            month_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
            print(f"\n  [SAVED] {csv_name} → {output_csv}")

        success_count = (month_df["log_scrape_status"] == "SUCCESS").sum()
        print(f"  Rows: {len(month_df)}  |  SUCCESS: {success_count}  |  FAILED: {len(month_df) - success_count}")
        grand_total_rows += len(month_df)
    else:
        print(f"  No rows collected for {csv_name}")

# ───────────────── SCRIPT END TIME ─────────────────

script_end      = time.time()
script_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
total_duration  = round(script_end - script_start, 2)

print("\n" + "═" * 70)
print("ALL DONE")
print(f"SCRIPT STARTED:    {script_start_str}")
print(f"SCRIPT ENDED:      {script_end_str}")
print(f"TOTAL TIME:        {total_duration}s  ({round(total_duration/60, 2)} min)")
print(f"TOTAL ROWS SAVED:  {grand_total_rows}")
print(f"OUTPUT DIR:        {OUTPUT_DIR}")
print("═" * 70)

[Output cleared — was a large execution log, removed to keep file size small]


## Part 5 ERROR and SKIPPED

**Repair pass — two jobs in one run.** Job 1 (`RETRY_MONTHS`): opens the already-scraped CSV of each listed month, picks out the rows whose `log_scrape_status` is `ERROR` or `SKIPPED`, re-scrapes only those URLs, and writes them to a separate retry folder under the same filename so they can be merged back later. Job 2 (`MISSING_MONTHS`): months that were never scraped at all are scraped in full from their original URL-list CSV and saved normally into the output folder. Parsing, retries, delays and logging are identical to Part 3/4.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import os
import glob
import json
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════
#  MALAY MAIL SCRAPER  —  TARGETED RETRY + MISSING MONTHS MODE
#  Mirrors the original scraper's request/parse/log logic exactly.
#
#  This version does TWO separate jobs in one run:
#
#  JOB 1 — RETRY_MONTHS
#    For each month listed in RETRY_MONTHS, open its EXISTING scraped
#    output CSV (in OUTPUT_DIR), find rows where log_scrape_status is
#    "ERROR" or "SKIPPED", re-scrape ONLY those URLs, and save the
#    results to a SEPARATE folder (RETRY_OUTPUT_DIR) using the SAME
#    filename as the month (e.g. malaymail_2015_05_scraped.csv) so you
#    can merge them back into the original files later.
#
#  JOB 2 — MISSING_MONTHS
#    For each month listed in MISSING_MONTHS, these were never scraped
#    at all. Read the original URL-list CSV from INPUT_DIR and do a
#    full scrape exactly like the original script, saving normally
#    into OUTPUT_DIR (not the retry folder, since there's nothing to
#    merge — this IS the first scrape for these months).
# ═══════════════════════════════════════════════════════════════════════

# ───────────────── INPUT / OUTPUT ─────────────────

# Folder with the ORIGINAL URL-list CSVs (malaymail_YYYY_MM.csv)
INPUT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\outputs_urls"

# Folder with the EXISTING scraped output CSVs (malaymail_YYYY_MM_scraped.csv)
# — this is where we READ FROM to find ERROR/SKIPPED rows for JOB 1
SCRAPED_OUTPUT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\scraped"

# Folder where JOB 1 (retry) results are saved — SEPARATE from originals
# so nothing gets overwritten; merge manually afterward.
RETRY_OUTPUT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\retry_scraped"

# Folder where JOB 2 (missing months) results are saved — normal location,
# same convention as the original full scraper.
OUTPUT_DIR = SCRAPED_OUTPUT_DIR

CHECKPOINT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001-20260617T104802Z-3-001\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\checkpoints"

# ───────────────── JOB 1: MONTHS TO RETRY (ERROR / SKIPPED rows only) ─────────────────

RETRY_MONTHS = [
    "2015_05", "2015_06",
    "2016_04", "2016_12",
    "2018_01", "2018_03",
    "2023_05",
    "2025_05", "2025_08",
]

# ───────────────── JOB 2: MONTHS MISSED ENTIRELY (full scrape) ─────────────────

MISSING_MONTHS = [
    "2019_08",
    "2021_04",
]

# ───────────────── RETRY / CHECKPOINT CONFIG ─────────────────

MAX_RETRIES      = 3
RETRY_DELAY      = 5
CHECKPOINT_EVERY = 50

# ───────────────── TEST MODE ─────────────────
# True  → only process the first month in each job, first 3 URLs each
# False → process everything listed above, in full
TEST_MODE = False

# ───────────────── REQUEST HEADERS ─────────────────

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

# ───────────────── STATUS CODE MESSAGES ─────────────────

STATUS_MESSAGES = {
    200: "OK",
    301: "Moved Permanently",
    302: "Found (Redirect)",
    403: "Forbidden",
    404: "Not Found",
    429: "Too Many Requests",
    500: "Internal Server Error",
    503: "Service Unavailable",
}

def get_status_msg(code):
    return STATUS_MESSAGES.get(code, "Unknown Status")

# ───────────────── MALAY MAIL PARSER  (unchanged from original) ─────────────────

def parse_article(html, url):
    soup = BeautifulSoup(html, "lxml")

    # ── TITLE ──
    title = ""
    h1 = soup.find("h1", class_="article-title")
    if h1:
        title = h1.get_text(" ", strip=True)
    elif soup.find("h1"):
        title = soup.find("h1").get_text(" ", strip=True)
    elif soup.title:
        title = soup.title.get_text(" ", strip=True).replace("| Malay Mail", "").strip()

    # ── PUBLISHED DATE + TIME ──
    published_date = ""
    published_time = ""
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date:
        raw = meta_date.get("content", "").strip()
        parts = raw.split(" ")
        if len(parts) >= 1:
            published_date = parts[0]
        if len(parts) >= 2:
            published_time = " ".join(parts[1:])

    # ── SECTION ──
    section = ""
    sec_div = soup.find("div", class_="article-section")
    if sec_div:
        sec_a = sec_div.find("a")
        if sec_a:
            section = sec_a.get_text(" ", strip=True)

    # ── AUTHOR ──
    author = ""
    meta_author = soup.find("meta", {"name": "author"})
    if meta_author:
        author = meta_author.get("content", "").strip()
    if not author:
        byline = soup.find("div", class_="article-byline")
        if byline:
            author = byline.get_text(" ", strip=True)

    # ── DESCRIPTION ──
    description = ""
    meta_desc = soup.find("meta", {"name": "description"})
    if meta_desc:
        description = meta_desc.get("content", "").strip()

    # ── KEYWORDS ──
    keywords = ""
    meta_kw = soup.find("meta", {"name": "keywords"})
    if meta_kw:
        keywords = meta_kw.get("content", "").strip()

    # ── IMAGE ──
    article_img = ""
    meta_img = soup.find("meta", {"property": "og:image"})
    if meta_img:
        article_img = meta_img.get("content", "").strip()

    # ── ARTICLE BODY TEXT ──
    paragraphs = []
    article_body = soup.find("div", class_="article-body")
    if article_body:
        for p in article_body.find_all("p"):
            text = p.get_text(" ", strip=True)
            if not text:
                continue
            bad_patterns = [
                "Follow us on",
                "Subscribe to",
                "You May Also Like",
                "Related Articles",
            ]
            if any(x in text for x in bad_patterns):
                continue
            paragraphs.append(text)
    else:
        for p in soup.find_all("p"):
            text = p.get_text(" ", strip=True)
            if text:
                paragraphs.append(text)

    paragraphs   = list(dict.fromkeys(paragraphs))
    article_text = "\n\n".join(paragraphs)

    return {
        "keyword":        "Malaysia",
        "published_date": published_date,
        "published_time": published_time,
        "section":        section,
        "author":         author,
        "title":          title,
        "description":    description,
        "keywords":       keywords,
        "article_text":   article_text,
        "article_img":    article_img,
        "url":            url,
    }

# ───────────────── CHECKPOINT SAVE ─────────────────

def save_checkpoint(rows, checkpoint_path):
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    pd.DataFrame(rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
    print(f"  [CHECKPOINT] {len(rows)} rows → {checkpoint_path}")


# ───────────────── CORE: SCRAPE A LIST OF URLS ─────────────────
# Shared by both JOB 1 (retry) and JOB 2 (missing months).
# Returns the list of result row dicts.

def scrape_url_list(urls, checkpoint_prefix, job_label):
    """
    Scrape every URL in `urls` using the exact same request/retry/parse
    logic as the original script. Returns a list of row dicts (one per
    URL) ready to be saved to a CSV.
    """
    all_rows       = []
    checkpoint_buf = []
    existing_ckpts = glob.glob(f"{checkpoint_prefix}_*.csv")
    checkpoint_idx = len(existing_ckpts) + 1
    total_urls     = len(urls)

    for url_idx, url in enumerate(urls, start=1):

        print("\n" + "=" * 70)
        print(f"  [{job_label}] [{url_idx}/{total_urls}] FETCHING")
        print(f"  URL: {url}")

        response          = None
        attempt           = 0
        request_start_str = ""
        request_end_str   = ""
        request_duration  = 0.0
        status_code       = ""
        status_msg        = ""
        redirected        = "No"
        redirect_hops     = ""
        final_url         = url
        req_headers_str   = ""
        resp_headers_str  = ""
        wait_time         = 0.0
        total_attempts    = 0
        error_msg         = ""
        scrape_status     = "SKIPPED"

        while attempt < MAX_RETRIES:

            attempt += 1
            print(f"    ATTEMPT: {attempt}/{MAX_RETRIES}")

            request_start     = time.time()
            request_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"    REQUEST START: {request_start_str}")

            try:
                response = requests.get(
                    url,
                    headers=HEADERS,
                    timeout=30,
                    allow_redirects=True
                )

                request_end      = time.time()
                request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                request_duration = round(request_end - request_start, 2)

                print(f"    REQUEST END:   {request_end_str}")
                print(f"    REQUEST TIME:  {request_duration}s")

                status_code = response.status_code
                status_msg  = get_status_msg(status_code)
                print(f"    STATUS: {status_code} — {status_msg}")

                if response.history:
                    redirected    = f"Yes ({len(response.history)} hop(s))"
                    redirect_hops = " | ".join(
                        f"{h.status_code} → {h.url}" for h in response.history
                    )
                    final_url = response.url
                    print(f"    REDIRECTED: {redirected}")
                    for i, hop in enumerate(response.history, 1):
                        print(f"      hop {i}: {hop.status_code} → {hop.url}")
                    print(f"    FINAL URL: {final_url}")
                else:
                    print(f"    REDIRECTED: No")

                req_headers_str  = json.dumps(dict(response.request.headers), ensure_ascii=False)
                resp_headers_str = json.dumps(dict(response.headers),         ensure_ascii=False)

                print("    REQUEST HEADERS:")
                for k, v in response.request.headers.items():
                    print(f"      {k}: {v}")
                print("    RESPONSE HEADERS:")
                for k, v in response.headers.items():
                    print(f"      {k}: {v}")

                if status_code == 200:
                    break

                if status_code in (429, 500, 503):
                    print(f"    Retryable {status_code}. Waiting {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"    Non-retryable {status_code}. Skipping.")
                    break

            except Exception as e:
                request_end      = time.time()
                request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                request_duration = round(request_end - request_start, 2)
                error_msg        = str(e)
                scrape_status    = "ERROR"
                print(f"    ERROR on attempt {attempt}: {error_msg}")
                print(f"    REQUEST TIME (before error): {request_duration}s")
                if attempt < MAX_RETRIES:
                    print(f"    Waiting {RETRY_DELAY}s before retry...")
                    time.sleep(RETRY_DELAY)

        total_attempts = attempt

        if response is not None and response.status_code == 200:
            html_text = response.text
            soup      = BeautifulSoup(html_text, "lxml")
            print(f"    TITLE CHECK: {soup.title}")
            print(f"    P TAGS: {len(soup.find_all('p'))}")
            article       = parse_article(html_text, url)
            print(f"    TITLE:        {article['title'][:100]}")
            print(f"    SECTION:      {article['section']}")
            print(f"    AUTHOR:       {article['author']}")
            print(f"    DATE:         {article['published_date']}")
            print(f"    TIME:         {article['published_time']}")
            print(f"    IMAGE FOUND:  {'Yes' if article['article_img'] else 'No'}")
            print(f"    TEXT CHARS:   {len(article['article_text'])}")
            scrape_status = "SUCCESS"
        else:
            article = {
                "keyword":        "Malaysia",
                "published_date": "",
                "published_time": "",
                "section":        "",
                "author":         "",
                "title":          "",
                "description":    "",
                "keywords":       "",
                "article_text":   "",
                "article_img":    "",
                "url":            url,
            }
            print(f"    SKIPPED (no valid response after {total_attempts} attempt(s))")

        wait_time = round(random.uniform(5, 10), 1)
        print(f"\n    WAITING {wait_time}s before next request...")
        time.sleep(wait_time)

        row = {
            **article,
            "log_request_start":      request_start_str,
            "log_request_end":        request_end_str,
            "log_request_duration_s": request_duration,
            "log_total_attempts":     total_attempts,
            "log_max_retries":        MAX_RETRIES,
            "log_wait_time_s":        wait_time,
            "log_status_code":        status_code,
            "log_status_msg":         status_msg,
            "log_redirected":         redirected,
            "log_redirect_hops":      redirect_hops,
            "log_final_url":          final_url,
            "log_request_headers":    req_headers_str,
            "log_response_headers":   resp_headers_str,
            "log_error_msg":          error_msg,
            "log_scrape_status":      scrape_status,
        }

        all_rows.append(row)
        checkpoint_buf.append(row)

        if len(checkpoint_buf) >= CHECKPOINT_EVERY:
            ckpt_path = f"{checkpoint_prefix}_{checkpoint_idx:04d}.csv"
            save_checkpoint(checkpoint_buf, ckpt_path)
            checkpoint_idx += 1
            checkpoint_buf  = []

    if checkpoint_buf:
        ckpt_path = f"{checkpoint_prefix}_{checkpoint_idx:04d}.csv"
        save_checkpoint(checkpoint_buf, ckpt_path)

    return all_rows


# ═══════════════════════════════════════════════════════════════════════
#  SCRIPT START
# ═══════════════════════════════════════════════════════════════════════

script_start     = time.time()
script_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"SCRIPT STARTED: {script_start_str}")

os.makedirs(RETRY_OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,       exist_ok=True)
os.makedirs(CHECKPOINT_DIR,   exist_ok=True)

grand_total_rows = 0


# ═══════════════════════════════════════════════════════════════════════
#  JOB 1 — RETRY ERROR / SKIPPED URLS FOR EXISTING MONTHS
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "█" * 70)
print("JOB 1 — RETRYING ERROR / SKIPPED URLS")
print("█" * 70)

retry_months_to_run = RETRY_MONTHS[:1] if TEST_MODE else RETRY_MONTHS

for month_idx, month in enumerate(retry_months_to_run, start=1):

    scraped_csv_name = f"malaymail_{month}_scraped.csv"
    scraped_csv_path = os.path.join(SCRAPED_OUTPUT_DIR, scraped_csv_name)

    print(f"\n[JOB1 {month_idx}/{len(retry_months_to_run)}] {month}")
    print(f"  Reading existing scraped file: {scraped_csv_path}")

    if not os.path.exists(scraped_csv_path):
        print(f"  ⚠️  WARNING: file not found — skipping {month}")
        continue

    try:
        scraped_df = pd.read_csv(scraped_csv_path)
    except Exception as e:
        print(f"  ⚠️  ERROR reading {scraped_csv_name}: {e} — skipping")
        continue

    if "log_scrape_status" not in scraped_df.columns or "url" not in scraped_df.columns:
        print(f"  ⚠️  Missing required columns in {scraped_csv_name} — skipping")
        continue

    # ── find rows that need retrying ──
    failed_mask = scraped_df["log_scrape_status"].isin(["ERROR", "SKIPPED"])
    failed_urls = scraped_df.loc[failed_mask, "url"].dropna().unique().tolist()

    print(f"  Total rows in file:     {len(scraped_df)}")
    print(f"  ERROR/SKIPPED rows:     {len(failed_urls)}")

    if not failed_urls:
        print(f"  ✅ No ERROR/SKIPPED rows for {month} — nothing to retry")
        continue

    if TEST_MODE:
        failed_urls = failed_urls[:3]
        print(f"  TEST MODE: limited to {len(failed_urls)} URLs")

    print(f"  URLs to retry: {len(failed_urls)}")

    checkpoint_prefix = os.path.join(CHECKPOINT_DIR, f"retry_{month}_ckpt")

    retried_rows = scrape_url_list(
        failed_urls,
        checkpoint_prefix,
        job_label=f"JOB1-{month}"
    )

    # ── save to SEPARATE retry folder, same filename as original ──
    retry_output_path = os.path.join(RETRY_OUTPUT_DIR, scraped_csv_name)

    if retried_rows:
        retry_df = pd.DataFrame(retried_rows)
        retry_df.to_csv(retry_output_path, index=False, encoding="utf-8-sig")

        success_count = (retry_df["log_scrape_status"] == "SUCCESS").sum()
        print(f"\n  [SAVED RETRY] {month} → {retry_output_path}")
        print(f"  Retried: {len(retry_df)}  |  Now SUCCESS: {success_count}  |  "
              f"Still failing: {len(retry_df) - success_count}")
        grand_total_rows += len(retry_df)
    else:
        print(f"  No rows produced for {month} retry")


# ═══════════════════════════════════════════════════════════════════════
#  JOB 2 — FULL SCRAPE FOR MONTHS MISSED ENTIRELY
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "█" * 70)
print("JOB 2 — FULL SCRAPE FOR MISSING MONTHS")
print("█" * 70)

missing_months_to_run = MISSING_MONTHS[:1] if TEST_MODE else MISSING_MONTHS

for month_idx, month in enumerate(missing_months_to_run, start=1):

    input_csv_name = f"malaymail_{month}.csv"
    input_csv_path = os.path.join(INPUT_DIR, input_csv_name)

    print(f"\n[JOB2 {month_idx}/{len(missing_months_to_run)}] {month}")
    print(f"  Reading URL list: {input_csv_path}")

    if not os.path.exists(input_csv_path):
        print(f"  ⚠️  WARNING: input file not found — skipping {month}")
        continue

    try:
        url_df = pd.read_csv(input_csv_path)
    except Exception as e:
        print(f"  ⚠️  ERROR reading {input_csv_name}: {e} — skipping")
        continue

    if "url" not in url_df.columns:
        print(f"  ⚠️  No 'url' column in {input_csv_name} — skipping")
        continue

    urls = url_df["url"].dropna().unique().tolist()

    if TEST_MODE:
        urls = urls[:3]
        print(f"  TEST MODE: limited to {len(urls)} URLs")

    print(f"  URLs to scrape: {len(urls)}")
    if not urls:
        print(f"  No URLs in {input_csv_name} — skipping")
        continue

    checkpoint_prefix = os.path.join(CHECKPOINT_DIR, f"missing_{month}_ckpt")

    scraped_rows = scrape_url_list(
        urls,
        checkpoint_prefix,
        job_label=f"JOB2-{month}"
    )

    # ── save normally into OUTPUT_DIR, same naming convention as before ──
    output_csv_name = f"malaymail_{month}_scraped.csv"
    output_csv_path = os.path.join(OUTPUT_DIR, output_csv_name)

    if scraped_rows:
        month_df = pd.DataFrame(scraped_rows)
        month_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

        success_count = (month_df["log_scrape_status"] == "SUCCESS").sum()
        print(f"\n  [SAVED] {month} → {output_csv_path}")
        print(f"  Rows: {len(month_df)}  |  SUCCESS: {success_count}  |  "
              f"FAILED: {len(month_df) - success_count}")
        grand_total_rows += len(month_df)
    else:
        print(f"  No rows collected for {month}")


# ───────────────── SCRIPT END TIME ─────────────────

script_end      = time.time()
script_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
total_duration  = round(script_end - script_start, 2)

print("\n" + "═" * 70)
print("ALL DONE")
print(f"SCRIPT STARTED:    {script_start_str}")
print(f"SCRIPT ENDED:      {script_end_str}")
print(f"TOTAL TIME:        {total_duration}s  ({round(total_duration/60, 2)} min)")
print(f"TOTAL ROWS SAVED:  {grand_total_rows}")
print(f"JOB 1 retry output: {RETRY_OUTPUT_DIR}")
print(f"JOB 2 full output:  {OUTPUT_DIR}")
if TEST_MODE:
    print("\n⚠️  TEST MODE was ON — only 1 month per job, max 3 URLs each")
    print("   Set TEST_MODE = False to process everything listed above")
print("═" * 70)

[Output cleared — was a large execution log, removed to keep file size small]
